In [43]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from pydantic import Field,BaseModel
from dotenv import load_dotenv
import operator

In [44]:
load_dotenv()

True

In [45]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [46]:
class EvaluationSchema(BaseModel):
    feedback:str = Field(description="detailed feedback for the essay")
    score:int = Field(description="Score out of 10",ge=0,le=10)


In [47]:
structured_model = model.with_structured_output(EvaluationSchema)

In [48]:
class UPSCState(TypedDict):
    essay_text: str
    cot_feedback:str
    doa_feedback:str
    lang_feedback:str
    final_feedback:str
    individual_score:Annotated[list[int],operator.add]
    final_score:int

In [49]:
def evaluate_lang(state:UPSCState)->UPSCState:
    prompt = f"write a detailed analysis on the language of the following essay and assign a score out of 10\n{state['essay_text']}"
    lang_feedback=structured_model.invoke(prompt)
    return {"lang_feedback":lang_feedback.feedback,"individual_score":[lang_feedback.score]}

In [50]:
def evaluate_thought(state:UPSCState)->UPSCState:
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay_text"]}'
    cot_feedback=structured_model.invoke(prompt)
    return {"cot_feedback":cot_feedback.feedback,"individual_score":[cot_feedback.score]}

In [51]:
def evaluate_analysis(state:UPSCState)->UPSCState:
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay_text"]}'
    doa_feedback=structured_model.invoke(prompt)
    return {"doa_feedback":doa_feedback.feedback,"individual_score":[doa_feedback.score]}

In [52]:
def final_eval(state:UPSCState)->UPSCState:
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["lang_feedback"]} \n depth of analysis feedback - {state["doa_feedback"]} \n clarity of thought feedback - {state["cot_feedback"]}'
    overall_feedback = model.invoke(prompt).content
    score = sum(state["individual_score"])/len(state["individual_score"])

    return {"final_feedback":overall_feedback,"final_score":score}

In [53]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_thought",evaluate_thought)
graph.add_node("evaluate_analysis",evaluate_analysis)
graph.add_node("evaluate_lang",evaluate_lang)
graph.add_node("final_eval",final_eval)

In [54]:
graph.add_edge(START,"evaluate_analysis")
graph.add_edge(START,"evaluate_thought")
graph.add_edge(START,"evaluate_lang")
graph.add_edge("evaluate_analysis","final_eval")
graph.add_edge("evaluate_thought","final_eval")
graph.add_edge("evaluate_lang","final_eval")
graph.add_edge("final_eval",END)
workflow = graph.compile()

In [55]:
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [56]:
initial_state={"essay_text":essay2}
workflow.invoke(initial_state)

{'essay_text': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad